# End-to-End Tutorial: HP35 Villin Headpiece (2RJY) Analysis with MolScope

This interactive tutorial demonstrates how to use MolScope's high-level `Study` API to perform a complete molecular dynamics analysis workflow on the canonical fast-folding **HP35 Villin Headpiece** (PDB: `2RJY`).

### Workflow Overview
1. **Environment Setup & Dataset Ingestion**: Load topology and trajectory files into an auditable `Study`.
2. **Periodic Boundary Imaging & Featurization**: Unfold and center coordinates on-the-fly; extract Cα contact matrices, DSSP secondary structure fractions, and sidechain rotamer dihedrals.
3. **Conformational Dimensionality Reduction**: Fit Variational Time-lagged Independent Component Analysis (TICA) and Contact Kernel PCA (CKPCA).
4. **State Discovery via Density Peak Advanced (DPA)**: Identify metastable conformational basins without specifying cluster counts in advance.
5. **Free Energy Surface (FES) Profiling**: Compute potential of mean force $\Delta G(x, y) = -k_B T \ln P(x, y)$ on the slowest kinetic coordinates.
6. **Explainable AI (Decision Tree Rules)**: Extract human-interpretable Boolean structural criteria explaining state transitions.
7. **3D Molecular Visualization**: Export hot-spot residue scores directly to PDB B-factors with PyMOL and ChimeraX command scripts.


## 1. System Overview and Environment Setup

The Villin Headpiece submodule (HP35, residues 13–76) is a canonical benchmark for protein folding dynamics:
- **Topology**: PDB structure `protein.pdb`
- **Trajectory**: 1,000 ns simulation trajectory `simulation_run1.xtc` (10,001 frames)

First, let's verify required imports and environment capabilities:

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from molscope.api import Study
from molscope.core.workspace import initialize_study

print(f"Python interpreter: {sys.executable}")
print("MolScope Study API successfully imported.")


## 2. Interactive Study Workflow

### Step 2.1: Initializing the Study and Adding the System

We initialize a stateful `Study` instance. The `Study` object manages:
- Provenance tracking and parameter hashing
- Caching of intermediate featurization matrices
- Output artifact directories and standardized subfolders (`figures/`, `tables/`, `3d_visualization/`)

In [ ]:
# Initialize output directory for the tutorial run
out_dir = Path("results/2rjy_tutorial")
out_dir.mkdir(parents=True, exist_ok=True)

study = Study(
    project_id="2rjy_villin_headpiece",
    out_dir=str(out_dir),
)

# Locate dataset: checks local data/2RJY directory first, then falls back to test fixtures
data_candidates = [
    Path("data/2RJY"),
    Path("../data/2RJY"),
    Path("../../data/2RJY"),
    Path("tests/fixtures"),
    Path("../tests/fixtures"),
    Path("../../tests/fixtures"),
]

loaded_system = False
for base in data_candidates:
    top_candidate = base / "protein.pdb"
    traj_candidate = base / "simulation_run1.xtc"
    if top_candidate.is_file() and traj_candidate.is_file():
        study.add_system(
            system_id="villin_headpiece",
            topology=str(top_candidate.resolve()),
            trajectories=[str(traj_candidate.resolve())],
        )
        print(f"Loaded 2RJY Villin Headpiece dataset from: {base}")
        loaded_system = True
        break

if not loaded_system:
    # Fallback to test fixtures for demonstration
    fixture_top = next((b / "1A22.pdb" for b in data_candidates if (b / "1A22.pdb").is_file()), None)
    fixture_traj = next((b / "1A22_traj.dcd" for b in data_candidates if (b / "1A22_traj.dcd").is_file()), None)
    if fixture_top and fixture_traj:
        study.add_system(
            system_id="demo_fixture",
            topology=str(fixture_top.resolve()),
            trajectories=[str(fixture_traj.resolve())],
        )
        print(f"Loaded demo fixture from: {fixture_top.parent}")
        print("To run the complete 1,000 ns 2RJY trajectory, place protein.pdb and simulation_run1.xtc in data/2RJY/")
    else:
        print("Warning: No trajectory files found on disk. Place 2RJY files in data/2RJY/.")


### Step 2.2: Periodic Boundary Imaging and Feature Plugins

Un-imaged coordinates wrap across periodic box boundaries, creating artificial covalent breaks and distorted inter-residue distance jumps.
We configure on-the-fly unwrapping and protein centering, along with 3 key biophysical feature plugins:
1. `protein.ca_distances_contacts`: Cα-Cα pairwise contact network (cutoff: 8.0 Å)
2. `protein.secondary_structure_dssp`: Helix, sheet, and coil fractions
3. `protein.sidechain_torsions`: Rotamer dihedral angles for core hydrophobic packing residues

In [ ]:
# Unfold and center on the protein to eliminate boundary jump artifacts
study.configure_imaging(unwrap=True, center_selection="protein", wrap=False)

# Configure feature plugins
study.configure_features([
    "protein.ca_distances_contacts",
    "protein.secondary_structure_dssp",
    "protein.sidechain_torsions",
])

# Set domain parameters
study.set_plugin_parameters("protein.ca_distances_contacts", {
    "selection": "protein and name CA",
    "contact_cutoff_angstrom": 8.0,
    "max_pairs": 3000,
})
study.set_plugin_parameters("protein.sidechain_torsions", {
    "residue_ids": [17, 20, 23, 27, 30, 35, 40, 50],
})

# Run extraction
study.run(extract_only=True)
features = study.get_features()
print(f"Extracted {features.shape[1]} features across {features.shape[0]} trajectory frames.")
display(features.head(3))


## 3. Conformational Dimensionality Reduction

High-dimensional contact and dihedral spaces obscure collective conformational transitions. We apply two complementary reduction methods:
1. **Variational TICA**: Finds linear combinations of coordinates that maximize autocorrelation at a defined lag time $\tau$, isolating the slowest kinetic processes.
2. **Contact Kernel PCA (CKPCA)**: Projects contact state vectors into a non-linear Reproducing Kernel Hilbert Space (RKHS) to capture curved conformational manifolds.

In [ ]:
# 1. Fit TICA (lag time = 2 frames = 1.0 ns at stride 5)
tica_res = study.run_tica(lag_time=2, components=5)
print("Slowest implied timescales (ps):", [round(t, 2) for t in tica_res.implied_timescales[:3]])

# 2. Fit Contact KPCA
ckpca_res = study.run_contact_kpca(n_components=5)
print("Contact KPCA explained variance ratio:", [round(v, 4) for v in ckpca_res.explained_variance_ratio[:3]])

# Plot projection onto the top 2 kinetic reaction coordinates
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(tica_res.coordinates.iloc[:, 0], tica_res.coordinates.iloc[:, 1], s=12, alpha=0.5, c="royalblue")
ax.set_xlabel("TIC 1 (Slowest Mode)")
ax.set_ylabel("TIC 2")
ax.set_title("HP35 Villin Headpiece: TICA Kinetic Embedding")
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


## 4. State Discovery via Density Peak Advanced (DPA)

Rather than assuming a pre-determined cluster count (as in $k$-means), MolScope uses **Density Peak Advanced (DPA)** clustering.
DPA identifies local probability density peaks in the kinetic reaction space and validates them with statistical significance thresholds ($z$-score $\ge 2.0$), preventing over-partitioning.

In [ ]:
# Cluster with Density Peak Advanced
dpa_res = study.cluster_dpa(coordinate_name="TICA", z_score=2.0)
print(f"Discovered {dpa_res.n_clusters} conformational basins:")
for state_id, pop in dpa_res.state_populations.items():
    frac = pop / len(features) * 100
    print(f"  State {state_id}: {pop} frames ({frac:.1f}%)")

# Visualize cluster assignments on TICA coordinates
plt.figure(figsize=(7, 5))
scatter = plt.scatter(
    tica_res.coordinates.iloc[:, 0],
    tica_res.coordinates.iloc[:, 1],
    c=dpa_res.labels,
    cmap="tab10",
    s=15,
    alpha=0.6,
)
plt.colorbar(scatter, label="DPA Cluster ID")
plt.xlabel("TIC 1")
plt.ylabel("TIC 2")
plt.title("DPA Metastable Basins in TICA Kinetic Space")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


## 5. Free Energy Surface (FES) Profiling

We compute the potential of mean force (PMF) $\Delta G(x, y) = -k_B T \ln P(x, y)$ on the slowest kinetic coordinates (TIC1 vs TIC2) at 300 K.
The surface is normalized such that the global free energy minimum has $\Delta G = 0\text{ kcal/mol}$.

In [ ]:
fes_img_path = out_dir / "fes_tic1_tic2.png"
fes_html_path = out_dir / "fes_tic1_tic2.html"

fes = study.plot_free_energy_landscape(
    dim1="TIC1",
    dim2="TIC2",
    temperature_kelvin=300.0,
    energy_units="kcal/mol",
    out_path=str(fes_img_path),
    interactive_html=str(fes_html_path),
)
print(f"Detected {len(fes.basins)} free energy minima.")
for i, b in enumerate(fes.basins):
    print(f"  Basin {i+1}: coordinates=({b.x:.2f}, {b.y:.2f}), free energy={b.free_energy:.2f} kcal/mol")


## 6. Explainable AI: Decision Tree Rule Extraction

To understand *what* structural transitions separate the discovered conformational basins, MolScope fits regularized Decision Trees.
Rather than complex black-box classifiers, MolScope extracts transparent Boolean rules with evaluated precision, recall, and support:

$$\text{IF } [\text{dist}(C_\alpha 35, C_\alpha 50) \le 6.8\text{ \AA}] \text{ AND } [\chi_1(\text{PHE 47}) \ge 65.2^\circ] \implies \text{Folded Native State (Purity 98.4\%)}

In [ ]:
# Extract boolean decision rules explaining each cluster
explainer = study.explain_clusters(max_depth=3, top_k_features=8)

print(f"Extracted {len(explainer.rules)} Boolean rules explaining cluster transitions:")
for rule in explainer.rules:
    print(f"\n--- State {rule.cluster_id} ({rule.cluster_name}) ---")
    print(f"  Support: {rule.support_fraction * 100:.1f}% | Purity: {rule.purity * 100:.1f}%")
    print(f"  Rule: {rule.simplified_rule_text()}")


## 7. 3D Hotspot Mapping for PyMOL and ChimeraX

MolScope projects feature importance scores directly onto the residue B-factors of a reference PDB structure.
This enables instant 3D visualization of allosteric hinges, active site residues, and folding nuclei:

In [ ]:
vis_dir = out_dir / "3d_visualization"
vis_dir.mkdir(parents=True, exist_ok=True)

scripts = study.export_3d(out_dir=str(vis_dir))
print("3D visualization exports generated:")
for key, path in scripts.items():
    print(f"  {key}: {path}")


### Viewing in Molecular Graphics

To inspect the residue importance spectrum in **PyMOL**:
```bash
pymol results/2rjy_tutorial/3d_visualization/view_scores.pml
```

Or in **UCSF ChimeraX**:
```bash
chimerax results/2rjy_tutorial/3d_visualization/view_scores.cxc
```

---